In [36]:
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

In [41]:
ticker = "ORWE.CA"

# Download historical data (daily OHLCV)
df = yf.download(ticker, start="2018-01-01")
df.columns = df.columns.get_level_values(0)

print(df.head())

[*********************100%***********************]  1 of 1 completed

Price          Close      High       Low      Open  Volume
Date                                                      
2018-01-01  5.323652  5.323652  5.323652  5.323652       0
2018-01-02  5.323652  5.323652  5.323652  5.323652       0
2018-01-03  5.272308  5.333278  5.224174  5.320442  142612
2018-01-04  5.339696  5.419920  5.297979  5.272308  496533
2018-01-07  5.339696  5.339696  5.339696  5.339696       0


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2045 entries, 2018-01-01 to 2026-04-21
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2045 non-null   float64
 1   High    2045 non-null   float64
 2   Low     2045 non-null   float64
 3   Open    2045 non-null   float64
 4   Volume  2045 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 95.9 KB


In [57]:
df.to_csv("orwe_data_raw.csv")

# Feature engineering

In [44]:
#Price based features
df['return_1d'] = df['Close'].pct_change()
df['return_5d'] = df['Close'].pct_change(5)
df['return_10d'] = df['Close'].pct_change(10)

In [45]:
df

Price,Close,High,Low,Open,Volume,return_1d,return_5d,return_10d
Date,,,,,,,,
2018-01-01,5.323652,5.323652,5.323652,5.323652,0,NaN,NaN,NaN
2018-01-02,5.323652,5.323652,5.323652,5.323652,0,0.000000,NaN,NaN
2018-01-03,5.272308,5.333278,5.224174,5.320442,142612,-0.009644,NaN,NaN
2018-01-04,5.339696,5.419920,5.297979,5.272308,496533,0.012782,NaN,NaN
2018-01-07,5.339696,5.339696,5.339696,5.339696,0,0.000000,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-04-15,22.930000,23.160000,22.860001,22.900000,1246039,0.001310,0.015051,0.044171
2026-04-16,22.799999,23.299999,22.650000,22.930000,1926558,-0.005669,0.022422,0.038724
2026-04-19,22.780001,22.980000,22.719999,22.799999,722377,-0.000877,0.006628,0.022442


In [46]:
#Trend indicators
df['ma_10'] = df['Close'].rolling(10).mean()
df['ma_50'] = df['Close'].rolling(50).mean()

#Relative postion to trend
df['ma_ratio_10'] = df['Close'] / df['ma_10']
df['ma_ratio_50'] = df['Close'] / df['ma_50']

In [50]:
#RSI indicator
delta = df['Close'].diff()

gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()

rs = gain / loss
df['rsi'] = 100 - (100 / (1 + rs))

In [51]:
#MACD indicator
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()

df['macd'] = ema_12 - ema_26
df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
df['macd_hist'] = df['macd'] - df['macd_signal']

In [52]:
#Volatility
df['volatility_10'] = df['Close'].pct_change().rolling(10).std()
df['volatility_20'] = df['Close'].pct_change().rolling(20).std()

In [53]:
#Volume based features
df['volume_avg_10'] = df['Volume'].rolling(10).mean()
df['volume_ratio'] = df['Volume'] / df['volume_avg_10']

In [54]:
df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

In [55]:
df.dropna(inplace=True)

In [56]:
df

Price,Close,High,Low,Open,Volume,return_1d,return_5d,return_10d,ma_10,ma_50,...,ma_ratio_50,rsi,macd,macd_signal,macd_hist,volatility_10,volatility_20,volume_avg_10,volume_ratio,target
Date,,,,,,,,,,,,,,,,,,,,,
2018-03-11,5.172830,5.262681,5.134323,5.156786,414339,0.003111,0.006871,0.073236,5.107368,5.174242,...,0.999727,58.754881,0.002184,-0.025990,0.028174,0.016413,0.015424,742887.3,0.557741,1
2018-03-12,5.233800,5.307606,5.137532,5.172830,2267233,0.011787,0.030322,0.048875,5.131756,5.172445,...,1.011862,61.594231,0.014021,-0.017988,0.032009,0.013342,0.015179,743148.3,3.050849,0
2018-03-13,5.233800,5.233800,5.233800,5.233800,0,0.000000,0.036214,0.015567,5.139778,5.170648,...,1.012214,74.235802,0.023135,-0.009763,0.032898,0.009054,0.014986,523094.9,0.000000,0
2018-03-14,5.150367,5.237009,5.073352,5.156785,829527,-0.015941,0.019695,0.000000,5.139778,5.168209,...,0.996548,73.913021,0.023356,-0.003139,0.026496,0.010628,0.015550,526482.4,1.575603,0
2018-03-15,5.079771,5.166412,4.973875,5.150368,1515634,-0.013707,-0.014935,-0.010006,5.134644,5.163011,...,0.983878,66.530646,0.017632,0.001015,0.016617,0.011458,0.015932,641410.4,2.362971,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-15,22.930000,23.160000,22.860001,22.900000,1246039,0.001310,0.015051,0.044171,22.484000,22.689600,...,1.010595,56.766919,0.035761,-0.053826,0.089587,0.008929,0.010708,755780.6,1.648678,0
2026-04-16,22.799999,23.299999,22.650000,22.930000,1926558,-0.005669,0.022422,0.038724,22.569000,22.702200,...,1.004308,51.879668,0.051627,-0.032735,0.084362,0.009383,0.010066,853061.8,2.258404,0
2026-04-19,22.780001,22.980000,22.719999,22.799999,722377,-0.000877,0.006628,0.022442,22.619000,22.713800,...,1.002915,57.983212,0.061875,-0.013813,0.075688,0.008591,0.009095,830327.0,0.869991,1


In [58]:
df.to_csv("orwe_data_features.csv")